# Session 14: RAG Capstone, Ingestion

**Notebook 1 of 2: Concepts** · the capstone kickoff, used during the live session and for revision afterwards (about 45 minutes on your own)

The next three sessions build one application: the **Internal Knowledge Assistant**, which answers questions about Orbital Logistics' runbooks, policies and postmortems, with sources. Session 11 taught every RAG building block. The capstone is about something different: putting them together into software that keeps working when the documents change, when files are broken, and when you run it for the hundredth time.

This session builds the first half, **ingestion**. By the end of this notebook you will be able to:

1. **Explain** the capstone's architecture, and justify where its module boundaries are.
2. **Design** idempotent ingestion: content hashes, stable chunk IDs, and replacing a document's chunks rather than adding to them.
3. **Predict** how Chroma treats repeated IDs, `None` metadata and collection metadata, and design around it.
4. **Decide** what counts as "unchanged", including the settings the index was built with.
5. **Isolate** failures per file, in the right order, and **log** them without leaking document contents.

> **Kernel:** the course environment, after `uv sync`. **Cost: nothing.** Everything runs offline: an in-memory Chroma database, a temporary folder, and a toy embedder. No model is downloaded.

The workbook then has you build the real thing in `projects/04-rag-capstone`. This notebook shows the problems and the tools; it deliberately doesn't assemble them into the solution.

## The engineering problem

Session 11's `handbook-search ingest` rebuilt its index from scratch every time. With 12 files and a local model, that took seconds, so nobody minded.

Now picture the real version. The operations team edits a few runbooks every day; the documentation folder has 3,000 files; embedding goes through a paid API, or takes 20 minutes on a laptop. A nightly job that re-embeds everything wastes time and money, and a job that only *adds* is worse: edited runbooks keep their old text in the index, and deleted policies keep answering questions.

So ingestion isn't a script you run once. It's a **sync job**: each run makes the index match the folder, doing only the work that changed, and surviving the broken file that someone saved half-way.

You've seen this pattern before, outside AI: `rsync` copies only changed files, `make` rebuilds only stale targets, Git knows which files changed by their content. Ingestion uses the same idea: **compare a fingerprint of each file with what the index recorded, and act on the difference.** Every file ends a run in one of five states: added, updated, unchanged, removed, or failed.

![The Internal Knowledge Assistant: architecture](../../../assets/diagrams/capstone-architecture.svg)

**How this works**

- `ingest()` in `ingestion.py` is the orchestrator: it decides *what* happens to each file, but it doesn't know how PDFs are parsed or how Chroma stores vectors. Each of those details lives in exactly one module, behind a small set of functions.
- `store.py` is the **only module that imports `chromadb`**. If you later move to another vector database, one file changes. The same reasoning made Session 09's `Triager` a protocol: callers depend on what a component does, not on the library behind it.
- `embedding.py` hides the model behind the `Embedder` protocol. The real embedder runs fastembed; the tests use a hashing embedder that is instant, free and deterministic, so the workbook's 72 tests run in seconds without downloading anything.
- The amber note is the heart of this session: unchanged files cost almost nothing, and one broken file never costs you the rest of the index.
- The dashed half is next week. Answering reads the index that ingestion writes, which is why the boundary between them (the `ChunkStore` and the `Chunk` model) has to be right now.

### Why these boundaries?

A useful test for a module boundary: **does each module have one reason to change?**

| Module | Changes when... | Doesn't change when... |
|---|---|---|
| `documents.py` | you support a new file type (`.docx`), or fix how PDF text is extracted | you switch vector database or model |
| `chunking.py` | you tune chunk size or splitting rules | you add a file type |
| `embedding.py` | you switch embedding model or provider | anything about files |
| `store.py` | you switch vector database | anything about files or models |
| `ingestion.py` | the *policy* changes: what to skip, retry or remove | any library changes |

In Java or C# you'd reach for interfaces and dependency injection here. Python gets the same decoupling with less ceremony: a `Protocol` describes the shape (`Embedder` has `model_id`, `dimensions` and `embed()`), any class with that shape fits, and `ingest(folder, store, embedder)` simply receives what it uses. There's no container and no registration: the CLI builds the real objects, and the tests build fakes.

**Engineering Insight:** boundaries also decide what you can test. Because `ingest()` receives its embedder, a test can pass one that counts calls, or one that fails on the third file. You'll see both in the workbook's tests.

## Part 1: Fingerprints

To know whether a file changed, you need a fingerprint of it. There are two candidates: the file's modification time, and a hash of its content.

In [ ]:
import hashlib
import os
import tempfile
import time
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="s14-"))
runbook = workdir / "vpn-outage.md"
runbook.write_text("# Runbook: VPN\n\nRestart the concentrator.\n", encoding="utf-8")


def sha256_of(path: Path) -> str:
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


before = (runbook.stat().st_mtime, sha256_of(runbook))
time.sleep(0.05)
os.utime(runbook)  # what `touch`, a Git checkout or a backup restore does
after = (runbook.stat().st_mtime, sha256_of(runbook))

print("modification time changed:", before[0] != after[0])
print("content hash changed:     ", before[1] != after[1])
print("hash:", after[1][:16], "...", len(after[1]), "hex characters")

The modification time changed although the content didn't: a Git checkout, a copy or a restore from backup would make every file look "changed". The content hash only changes when the bytes do. `hashlib.file_digest` (Python 3.11+) reads the file in blocks, so it works for a 2 GB PDF without loading it into memory.

**Predict:** the next cell saves the same text with Windows line endings (`\r\n`). Will the hash change?

In [ ]:
same_text_crlf = workdir / "vpn-outage-crlf.md"
same_text_crlf.write_bytes(runbook.read_bytes().replace(b"\n", b"\r\n"))

print("LF  :", sha256_of(runbook)[:16])
print("CRLF:", sha256_of(same_text_crlf)[:16])

**Explain:** it changes, because the hash covers *bytes*, not meaning. If a teammate's Git converts line endings, every Markdown file is re-embedded once. That's the right trade-off: a false "changed" costs one re-embedding, while a false "unchanged" leaves wrong answers in the index indefinitely. **Hashes err on the safe side; modification times err on both sides.**

| Fingerprint | Says "changed" when nothing did | Says "unchanged" when something did | Cost |
|---|---|---|---|
| Modification time | Checkout, copy, restore, `touch` | Tools that preserve times (`cp -p`, unzip) | Free |
| Size | Never | Any same-size edit (a typo fix: `P2` to `P1`) | Free |
| SHA-256 of the bytes | Line endings, re-saving with another tool | Practically never | Reads every byte once |

**Engineering Insight:** some systems check the modification time first and hash only files whose time changed. That's an optimisation for millions of files. For thousands, hashing everything takes a second, and it's one less thing to get wrong.

## Part 2: Running ingestion twice

Fingerprints tell you *which* files changed. The other half of the problem is what the vector store does when you write the same document again. Let's find out with a small in-memory Chroma collection and a toy embedder.

In [ ]:
import uuid

import chromadb
import numpy as np
from chromadb.config import Settings

client = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False))


def toy_embed(texts: list[str]) -> np.ndarray:
    """Counts of a few words: enough to tell runbooks apart, and free."""
    vocabulary = ["vpn", "restart", "database", "replica", "certificate", "backup"]
    rows = [[text.lower().count(word) + 0.01 for word in vocabulary] for text in texts]
    return np.array(rows, dtype=np.float32)


def fresh_collection(name: str):
    if name in [c.name for c in client.list_collections()]:
        client.delete_collection(name)
    return client.create_collection(
        name, configuration={"hnsw": {"space": "cosine"}}, embedding_function=None
    )


chunks = ["Restart the VPN concentrator.", "Check the VPN certificate.", "Call the network desk."]
naive = fresh_collection("naive")
for run in (1, 2):
    naive.add(
        ids=[str(uuid.uuid4()) for _ in chunks],  # a new random ID every time
        embeddings=toy_embed(chunks),
        documents=chunks,
        metadatas=[{"source": "runbooks/vpn-outage.md"} for _ in chunks],
    )
    print(f"after run {run}: {naive.count()} chunks")

hits = naive.query(query_embeddings=toy_embed(["vpn restart"]), n_results=3)
print("top 3 for 'vpn restart':", hits["documents"][0])

With random IDs, every run adds another copy. Search results fill up with duplicates of the same passage, crowding out everything else, and the index grows without limit.

The obvious fix is a **stable ID**: the same file and position always produce the same ID, such as `runbooks/vpn-outage.md#0`. Then the second run writes the same IDs again.

**Predict:** with stable IDs, the runbook is edited (*"Restart the VPN concentrator"* becomes *"Fail over to the secondary concentrator"*) and ingested again with `add`. What does the index hold afterwards?

In [ ]:
def stable_ids(source: str, count: int) -> list[str]:
    return [f"{source}#{index}" for index in range(count)]


source = "runbooks/vpn-outage.md"
stable = fresh_collection("stable")
stable.add(
    ids=stable_ids(source, 3),
    embeddings=toy_embed(chunks),
    documents=chunks,
    metadatas=[{"source": source}] * 3,
)

edited = ["Fail over to the secondary concentrator.", *chunks[1:]]
stable.add(
    ids=stable_ids(source, 3),
    embeddings=toy_embed(edited),
    documents=edited,
    metadatas=[{"source": source}] * 3,
)
print("count:", stable.count())
print("chunk #0 says:", stable.get(ids=[f"{source}#0"])["documents"])

**Explain:** no error, no duplicate, and **no update either**. In Chroma 1.5, `add` with an ID that already exists is silently ignored, so the edit never reaches the index. A run that "succeeds" every night can serve last month's runbook forever. Silent no-ops like this are why the workbook's tests check what the store *contains*, not just that ingestion ran without an exception.

`upsert` ("update or insert") writes the new text over the old. But try it on a document that got **shorter**:

In [ ]:
stable.upsert(
    ids=stable_ids(source, 3),
    embeddings=toy_embed(edited),
    documents=edited,
    metadatas=[{"source": source}] * 3,
)
print("after upsert, chunk #0:", stable.get(ids=[f"{source}#0"])["documents"])

shorter = ["Fail over to the secondary concentrator."]  # someone cut the runbook down
stable.upsert(
    ids=stable_ids(source, 1),
    embeddings=toy_embed(shorter),
    documents=shorter,
    metadatas=[{"source": source}],
)
left = stable.get(where={"source": source})
print(f"after the shorter version: {stable.count()} chunks")
for chunk_id, text in sorted(zip(left["ids"], left["documents"])):
    print(f"  {chunk_id}: {text}")

The runbook now has one chunk, but the index still holds chunks `#1` and `#2` from the old version. They'll keep being retrieved, and cited, as if they were part of the current runbook. **Upserting by position handles edits, not removals.**

What you actually want for a changed file is: *make these chunks the source's only chunks*. Chroma gives you the pieces: `get(where={"source": ...})` finds a source's chunks, and `delete(where=...)` or `delete(ids=...)` removes them. Putting them together, in an order that's safe when something fails half-way, is your `ChunkStore.replace()` in the workbook.

**Challenge:** before you write it, sketch two orders: "delete the old chunks, then add the new ones" versus "upsert the new ones, then delete what's left over". For each, what does a search see if the process crashes between the two steps?

## Part 3: Metadata that Chroma won't take, or quietly drops

Every chunk carries metadata: source, title, section, page, index and content hash. Some of it is legitimately missing. A Markdown chunk has no page, and a PDF chunk has no section, so in Python those fields are `None`.

In [ ]:
meta_demo = fresh_collection("metadata")
markdown_chunk = {"source": "runbooks/vpn-outage.md", "section": "Diagnose", "page": None}

try:
    meta_demo.add(
        ids=["a#0"], embeddings=toy_embed(["x"]), documents=["x"], metadatas=[markdown_chunk]
    )
except Exception as error:  # noqa: BLE001  (showing which error Chroma raises)
    print("add:", type(error).__name__, "-", error)

meta_demo.upsert(
    ids=["a#0"], embeddings=toy_embed(["x"]), documents=["x"], metadatas=[markdown_chunk]
)
print("upsert stored:", meta_demo.get(ids=["a#0"])["metadatas"])

Two different behaviours for the same input. `add` rejects `None` with a `TypeError`. `upsert` accepts it, because in an update `None` means "remove this key", and stores the chunk **without** a `page` key. Reading it back, `metadata["page"]` raises `KeyError`, while `metadata.get("page")` gives `None`.

The robust approach doesn't depend on which method you happen to call: **leave `None` fields out when you write, and treat a missing key as `None` when you read.** Then a `Chunk` read back from the store equals the `Chunk` you wrote, which is exactly what the workbook's store tests compare.

The collection itself has metadata too, and it's the natural place to record how the index was built. Two more surprises are waiting there:

In [ ]:
built = client.get_or_create_collection(
    "configured",
    metadata={"embedding_model": "model-a", "chunk_size": 800},
    embedding_function=None,
)
again = client.get_or_create_collection(
    "configured",
    metadata={"embedding_model": "model-b", "chunk_size": 500},
    embedding_function=None,
)
print("second get_or_create sees:", again.metadata)

again.modify(metadata={"embedding_model": "model-b"})
print("after modify:             ", client.get_collection("configured").metadata)

- `get_or_create_collection` **ignores** the metadata you pass when the collection already exists. It doesn't tell you that the index was built with `model-a` while you're about to add `model-b` vectors: you have to compare the recorded values yourself.
- `modify(metadata=...)` **replaces** the whole dictionary. `chunk_size` is gone, not kept.

**Common Mistake:** mixing vectors from two embedding models in one collection. Nothing crashes (if the dimensions happen to match), but similarity between vectors from different models is meaningless, and retrieval quietly gets worse. Session 11's store refused to load an index built with another model; the capstone's store refuses to *open* one.

## Part 4: What does "unchanged" mean?

Here's a trap that content hashes walk straight into. Suppose the index was built with 800-character chunks, and you change the setting to 300 to get more precise retrieval. You run ingestion again.

In [ ]:
recorded = {"runbooks/vpn-outage.md": sha256_of(runbook)}  # what the index remembers
settings_then = {"chunk_size": 800}
settings_now = {"chunk_size": 300}

for path in [runbook]:
    source = f"runbooks/{path.name}"
    if recorded.get(source) == sha256_of(path):
        print(f"{source}: unchanged, skipped")
print("chunk size changed:", settings_then != settings_now, "- but nothing was re-chunked")

Every hash matches, so every file is skipped, and the index keeps its 800-character chunks while your settings say 300. Nothing fails; it's just silently not what you configured. The same happens when you switch embedding model, or fix a bug in your PDF text extraction.

**The content hash answers "did the file change?", not "would processing it now give the same result?".** The second question depends on everything that shapes the output:

| Input to ingestion | Covered by the content hash? | How the capstone handles it |
|---|---|---|
| The file's bytes | Yes | Re-process the file |
| Chunk size and overlap | No | Recorded in the index; a different value is refused until you rebuild |
| Embedding model (and its dimensions) | No | Same |
| Your loader and chunker code | No | Rebuild by hand after changing them; an extension challenge is to record a pipeline version |

The capstone records an `IndexConfig` (model, dimensions, chunk size, overlap) in the collection's metadata. Opening the index with a different one raises `IndexMismatchError`, and `knowledge-assistant ingest --rebuild` starts again. Refusing is deliberately blunt: a partial re-chunk would leave an index with two chunk sizes in it.

![Incremental ingestion: what happens to each file](../../../assets/diagrams/capstone-ingestion-flow.svg)

**How this works**

- A run starts from two lists: what's on disk, from `discover()`, and what the index holds, from `store.sources()`, where each source maps to the hash its chunks were built from. The comparison between them drives everything else.
- **Unchanged** files are hashed and nothing more: no reading, no chunking, no embedding. A second run over an unchanged folder makes zero embedding calls, and the workbook's tests count them.
- A file that fails to load (corrupt PDF, not UTF-8, empty, scanned) is **reported and skipped**. The index keeps whatever it held for that file before: a half-saved PDF shouldn't erase the last good version.
- **Embed before you replace.** If the embedder fails, an exception stops the run, but the file being processed still has its previous chunks, and everything finished before it stays done. The next run picks up exactly where this one stopped, because finished files now hash as unchanged.
- Sources in the index but not on disk are **removed**. A rename is a removal plus an addition, which is correct: the old source name must stop appearing in citations.
- The amber box is Part 4: the hash covers the file only, so the configuration is checked separately.

## Part 5: One broken file, five hundred good ones

Real document folders contain broken files: a PDF that was half-uploaded, a Markdown file saved as Latin-1 by an old editor, an empty placeholder, a scanned image with no text. Ingestion should report them and carry on.

The tempting way to "carry on" is to catch everything. Here's why that's a trap. This little loop has a bug of its own:

In [ ]:
import logging


def load(path: Path) -> str:
    text = path.read_text(encoding="utf-8")
    if not text.strip():
        raise ValueError(f"{path.name}: empty")
    return text


def count_words(text: str) -> int:
    return len(text.split())


def process_everything(paths: list[Path]) -> dict[str, str]:
    failures = {}
    for path in paths:
        try:
            text = load(path)
            words = count_words(text, path)  # bug: an extra argument
        except Exception as error:  # noqa: BLE001  (the mistake this cell demonstrates)
            failures[path.name] = str(error)
    return failures


docs = workdir / "docs"
docs.mkdir(exist_ok=True)
(docs / "good-1.md").write_text("# One\n\nFine.", encoding="utf-8")
(docs / "good-2.md").write_text("# Two\n\nAlso fine.", encoding="utf-8")
(docs / "empty.md").write_text("", encoding="utf-8")

for name, reason in process_everything(sorted(docs.iterdir())).items():
    print(f"failed {name}: {reason}")

**Explain:** all three files "failed", but only one of them is broken. The other two failures are a bug in our own code, a `TypeError`, reported as if the documents were at fault. In production this looks like "ingestion ran, 500 files failed, must be bad data", and the bug survives for weeks.

The fix is to catch **the errors that mean "this document is bad"**, and nothing else. The capstone has its own exception for that, `DocumentError`: the loader raises it, with a reason, when a file isn't UTF-8, isn't a readable PDF, or has no text. `ingest()` catches `DocumentError` per file. Anything else, whether a bug in your code or an embedding service that's down, propagates and stops the run, loudly.

**Modify:** change `except Exception` to `except ValueError` and run the cell again. The empty file is reported, and the `TypeError` now stops the run, pointing at the real bug.

**Common Mistake:** converting library errors to your own error too broadly. pypdf raises `PdfReadError` (and subclasses such as `PdfStreamError` and `EmptyFileError`) for corrupt files; that's the one to translate into `DocumentError`. Translating `Exception` would turn your own bugs into "corrupt PDF" again.

## Part 6: Logging without leaking

Ingestion runs unattended, so its logs are how you find out what happened. They're also a classic leak: logs are copied to central systems, kept for months (the corpus's own data-retention policy says 90 days), and read by people who aren't allowed to read every document.

The rule: **log identifiers and counts, never contents.** A source like `policies/access-requests.md` is fine. The chunk text isn't, and neither is the full path, which often contains a username (`C:\Users\jane.doe\...`).

How do you *test* that? Put a unique, recognisable string (a **canary**) in a document, run the code with logging captured, and assert the canary never appears. The workbook's tests do exactly this. Here's the technique, catching a leak that isn't even in our own code:

In [ ]:
import io

captured = io.StringIO()
handler = logging.StreamHandler(captured)
logging.getLogger().addHandler(handler)
logging.getLogger().setLevel(logging.WARNING)

from pypdf import PdfReader  # noqa: E402
from pypdf.errors import PdfReadError  # noqa: E402

canary = "CANARY-5e1f"
broken_pdf = workdir / "broken.pdf"
broken_pdf.write_bytes(f"{canary} is the start of this file, not a PDF header".encode())
try:
    PdfReader(broken_pdf)
except PdfReadError as error:
    print("pypdf raised:", type(error).__name__)

logging.getLogger().removeHandler(handler)
print("captured log:", captured.getvalue().strip())
print("canary leaked:", canary[:5] in captured.getvalue())

pypdf logs a warning containing **the first five bytes of the file** when the header is wrong. Five bytes is harmless here, but it shows the point: libraries log too, and their messages can contain data you'd never log yourself. The capstone's provided CLI raises pypdf's log level to `ERROR` for that reason, and reports bad files through its own report instead.

**Engineering Insight:** put the canary in every place that data passes through: the text of a good file, a broken file, an error message. The workbook's logging test uses a good Markdown file, a non-UTF-8 file and a fake PDF, all containing the same canary, and checks the full log at `DEBUG` level.

**Run:** in Python's `logging`, pass values as arguments (`logger.info("added %s", source)`) instead of formatting them first (`logger.info(f"added {source}")`). The message is only built if the record is actually emitted, and log tools can group records by their message template.

## Common mistakes

| Mistake | What happens | Prevention |
|---|---|---|
| Random chunk IDs | Every run duplicates the index | IDs from source and position: `source#index` |
| `add` with existing IDs | Edits silently never reach the index | Replace a changed source's chunks completely |
| `upsert` alone | A shortened document leaves stale chunks behind | Delete the source's chunks beyond the new ones |
| Modification time as the fingerprint | Checkouts re-embed everything; some edits are missed | SHA-256 of the bytes |
| Only the content hash | Changing the model or chunk size changes nothing | Record the configuration; refuse or rebuild |
| `None` in Chroma metadata | `add` raises `TypeError`; `upsert` drops the key | Leave `None` out; read with `.get()` |
| `except Exception` around each file | Your bugs are reported as bad documents | Catch your own `DocumentError` only |
| Deleting old chunks before embedding | An embedder outage erases the document | Embed first, then replace |
| Logging chunk text or full paths | Documents and usernames leak into log storage | Log sources and counts; test with a canary |
| Forgetting deletions | Removed policies keep answering questions | Remove sources that are no longer on disk |

## Key takeaways

- Ingestion is a **sync job**: each run makes the index match the folder, and its work is proportional to what changed.
- A **content hash** says whether a file changed; the **recorded configuration** says whether the index can still be updated.
- Stable IDs prevent duplicates, but correctness needs **replace**, not add or upsert: edits, shrinking documents and deletions all have to land.
- Vector databases have quiet behaviours (ignored `add`s, dropped `None` keys, ignored or replaced collection metadata). Test what the store **contains**.
- Catch the errors that mean "bad document", and let everything else stop the run. **Embed before you replace**, so a failure never leaves a document half-indexed.
- Logs carry identifiers and counts, never contents; prove it with a **canary**.

## Further reading

- [`hashlib.file_digest`](https://docs.python.org/3.13/library/hashlib.html#hashlib.file_digest), the Python 3.13 documentation.
- [Chroma: adding data](https://docs.trychroma.com/docs/collections/add-data), [updating data](https://docs.trychroma.com/docs/collections/update-data) and [deleting data](https://docs.trychroma.com/docs/collections/delete-data).
- [Logging HOWTO](https://docs.python.org/3.13/howto/logging.html), especially "Optimization" for why `%s` arguments beat f-strings in log calls.
- The session's [references](../resources/references.md) list more, including the LangChain text splitters.